In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_bt = pd.read_csv("files/high_cosine_similarity_records.csv",
                    usecols=["translated_welsh", "cefr_level"])\
         .rename(columns={"translated_welsh": "text"})

In [3]:
# Add missing columns
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [4]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
0,"Ac mae'r tywysog yn mynd i ffwrdd, yn ddryslyd.",A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,"Dyma, i mi, y mae'r cariad yn drist a'r lleafa...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Roedd y pumed yn rhyfedd iawn.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,"Nid A oedd gofal llawer ar gyfer y nodau, roed...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,"I fod yn onest, erbyn hyn doeddwn i ddim wir w...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
1060,Gwnaeth farw ar 6 Hydref Hydref Hydref Hydref ...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1061,Y fersiwn cyntaf i' r fersiwn cyntaf.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1062,Rhestr o wledydd eraill gan boblogaeth,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1063,"Roedd hi'n unig blentyn, geni yn Llundain.",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [5]:
# Count how many samples are labeled A1 and A2
df_bt["cefr_level"].value_counts()

cefr_level
A2    914
A1    151
Name: count, dtype: int64

In [6]:
# Load Welsh CEFR dataset from HuggingFace
ds_welsh = load_dataset("UniversalCEFR/learn_welsh_cy")["train"].to_pandas()

In [7]:
ds_welsh

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,..."
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ..."
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b..."
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D..."
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...
...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?


In [11]:
df_merged = pd.concat([ds_welsh, df_bt], ignore_index=True)
df_merged = df_merged.drop_duplicates(subset="text", keep="first").sample(frac=1, random_state=42)

In [12]:
df_merged

,title,lang,source_name,format,category,cefr_level,license,text
439,Uned 12 - na,cy,mynediad-de-learnwelsh,sentence-level,reference,A1,public,Mae John yn ddwy oed.
2125,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A1,CC-BY-SA,Mae fy ngŵr yn gweithio'n galed.
613,Uned 20 - na,cy,mynediad-de-learnwelsh,sentence-level,reference,A1,public,Beth yw dy ebost di?
1107,Uned 9 - am,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,'Dyn ni wedi bod yn poeni amdanoch chi.
2115,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A1,CC-BY-SA,Mae hi eisiau chwarae cardiau.
...,...,...,...,...,...,...,...,...
1642,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A2,CC-BY-SA,"""Roeddwn yn gwybod ei fod wedi mynd unwaith bo..."
1098,Uned 9 - am,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Beth am Bethan?
1133,Uned 10 - Wnei di?,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Wnei di helpu gyda'r gwaith?
1297,Uned 17 -na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Caerdydd yw'r ddinas fwya swnllyd.


In [13]:
df_merged["cefr_level"].value_counts()

cefr_level
A2    1520
A1     913
Name: count, dtype: int64

In [18]:
hf_dataset=Dataset.from_pandas(df_merged.reset_index(drop=True))

In [19]:
hf_dataset

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 2433
})

In [20]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in hf_dataset["cefr_level"]])

In [21]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [22]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [23]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [24]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [25]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(hf_dataset, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = hf_dataset.select(train_idx)
    ds_val = hf_dataset.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_DA/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 487/487 [00:00<00:00, 9089.20 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_22452\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.879900,0.640939,0.675565,0.601405,0.722155,0.675565,0.820513,0.174863,0.288288,0.662946,0.976974,0.789894,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.539000,0.424179,0.813142,0.809029,0.812169,0.813142,0.802632,0.666667,0.728358,0.817910,0.901316,0.857590,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.373700,0.383180,0.848049,0.847525,0.847272,0.848049,0.807910,0.781421,0.794444,0.870968,0.888158,0.879479,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 2...


Map: 100%|██████████| 487/487 [00:00<00:00, 13953.98 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_22452\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Map: 100%|██████████| 486/486 [00:00<00:00, 15919.53 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_22452\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.740200,0.519773,0.736626,0.709392,0.751349,0.736626,0.800000,0.395604,0.529412,0.722222,0.940789,0.817143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.398600,0.518401,0.790123,0.793484,0.820820,0.790123,0.665289,0.884615,0.759434,0.913934,0.733553,0.813869,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.216100,0.430533,0.851852,0.852315,0.853046,0.851852,0.792553,0.818681,0.805405,0.889262,0.871711,0.880399,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 486/486 [00:00<00:00, 12626.48 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_22452\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.734800,0.519750,0.744856,0.748594,0.790868,0.744856,0.611538,0.873626,0.719457,0.898230,0.667763,0.766038,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.406800,0.531681,0.765432,0.734658,0.812469,0.765432,0.947368,0.395604,0.558140,0.731707,0.986842,0.840336,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.247700,0.424994,0.841564,0.842695,0.845460,0.841564,0.766497,0.829670,0.796834,0.892734,0.848684,0.870152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [26]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_DA/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [27]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [28]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.847272  0.848049  0.847525  0.807910  0.781421  0.794444   
1        2        0.880571  0.880903  0.879708  0.874251  0.797814  0.834286   
2        3        0.857541  0.858316  0.857741  0.823864  0.792350  0.807799   
3        4        0.853046  0.851852  0.852315  0.792553  0.818681  0.805405   
4        5        0.845460  0.841564  0.842695  0.766497  0.829670  0.796834   
5  Average        0.856778  0.856137  0.855997  0.813015  0.803987  0.807754   

         A2                      ...   B1        B2                    C1  \
  Precision    Recall        F1  ...   F1 Precision Recall   F1 Precision   
0  0.870968  0.888158  0.879479  ...  0.0       0.0    0.0  0.0       0.0   
1  0.884375  0.930921  0.907051  ...  0.0       0.0    0.0  0.0       0.0   
2  0.877814  0.898026  0.887805  ...  0.0       0.0    0.0  0.0       0.0   
3  0.889262  0.871711  0.880399  ...  0.0       0.0    0.0  0.0       0.0   
4  0.892734  0.848684  0.870152  ...  0.0       0.0    0.0  0.0       0.0   
5  0.883030  0.887500  0.884977  ...  0.0       0.0    0.0  0.0       0.0   

                     C2              
  Recall   F1 Precision Recall   F1  
0    0.0  0.0       0.0    0.0  0.0  
1    0.0  0.0       0.0    0.0  0.0  
2    0.0  0.0       0.0    0.0  0.0  
3    0.0  0.0       0.0    0.0  0.0  
4    0.0  0.0       0.0    0.0  0.0  
5    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]